<a href="https://colab.research.google.com/github/Amazeble/chatterbox-finetuning/blob/main/colab/chatterbox-finetuning-colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook guides you through fine-tuning the Chatterbox TTS model.

In [ ]:
#@title ⚙️ STEP 1: Configuration

#@markdown ### 📂 Path Settings
project_name = "Elise" #@param {type:"string"}
source_path = '/content/drive/MyDrive/MyTTSDataset/' #@param {type:"string"}
dest_path = '/content/chatterbox-finetuning/' #@param {type:"string"}
model_dir = "./pretrained_models" #@param {type:"string"}

#@markdown These paths will be auto-resolved based on project_name:
#@markdown - Dataset: `{dest_path}MyTTSDataset/{project_name}/`
#@markdown - Output: `{dest_path}chatterbox_output/{project_name}/`
csv_path = f"./MyTTSDataset/{project_name}/metadata.csv" #@param {type:"string"}
wav_dir = f"./MyTTSDataset/{project_name}/wavs" #@param {type:"string"}
preprocessed_dir = f"./MyTTSDataset/{project_name}/preprocess" #@param {type:"string"}
output_dir = f"./chatterbox_output/{project_name}" #@param {type:"string"}

#@markdown ### 🔊 Inference Settings
is_inference = False #@param {type:"boolean"}
inference_prompt_path = "./speaker_reference/2.wav" #@param {type:"string"}
inference_test_text = "Merhaba, sesimi gelis\u0327tirmem olduk\u00E7a uzun zaman ald\u0131 ve s\u0131mdi sahip oldu\u011Fuma g\u00F6re, sessiz kalmayaca\u011F\u0131m." #@param {type:"string"}

#@markdown ### 📦 Dataset Format Settings
ljspeech = True #@param {type:"boolean"}
json_format = False #@param {type:"boolean"}
preprocess = "True" #@param ["True", "False", "auto"]

#@markdown ### 🚀 Training Mode Settings
is_turbo = True #@param {type:"boolean"}
is_lora = True #@param {type:"boolean"}
is_merge_lora = True #@param {type:"boolean"}

#@markdown ### 📐 LoRA Parameters
lora_r = 128 #@param {type:"integer"}
lora_alpha = 256 #@param {type:"integer"}

#@markdown ### 🧠 Hyperparameters
batch_size = 8 #@param {type:"slider", min:1, max:16, step:1}
grad_accum = 4 #@param {type:"slider", min:1, max:16, step:1}
learning_rate = 1e-4 #@param {type:"number"}
num_epochs = 10 #@param {type:"slider", min:1, max:50, step:1}
save_steps = 500 #@param {type:"integer"}
save_total_limit = 5 #@param {type:"integer"}
dataloader_num_workers = 8 #@param {type:"integer"}

#@markdown ### 🔒 Constraints
start_text_token = 255 #@param {type:"integer"}
stop_text_token = 0 #@param {type:"integer"}
max_text_len = 256 #@param {type:"integer"}
max_speech_len = 850 #@param {type:"integer"}
prompt_duration = 3.0 #@param {type:"number"}

import os
import json
from IPython.display import clear_output

# Handle derived static variables
new_vocab_size = 52260 if is_turbo else 2454
turbo_lora_target_modules = ["c_attn", "c_proj", "c_fc", "spkr_enc"]
lora_target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj", "spkr_enc"]
lora_modules_to_save = ["text_emb", "text_head"]

# Pack all parameters into a clean dictionary layout
config_data = {
    "source_path": source_path,
    "dest_path": dest_path,
    "model_dir": model_dir,
    "wav_dir": wav_dir,
    "output_dir": output_dir,
    "is_inference": is_inference,
    "inference_prompt_path": inference_prompt_path,
    "inference_test_text": inference_test_text,
    "ljspeech": ljspeech,
    "json_format": json_format,
    "preprocess": preprocess,
    "is_turbo": is_turbo,
    "is_lora": is_lora,
    "is_merge_lora": is_merge_lora,
    "lora_r": lora_r,
    "lora_alpha": lora_alpha,
    "turbo_lora_target_modules": turbo_lora_target_modules,
    "lora_target_modules": lora_target_modules,
    "lora_modules_to_save": lora_modules_to_save,
    "new_vocab_size": new_vocab_size,
    "batch_size": batch_size,
    "grad_accum": grad_accum,
    "learning_rate": learning_rate,
    "num_epochs": num_epochs,
    "save_steps": save_steps,
    "save_total_limit": save_total_limit,
    "dataloader_num_workers": dataloader_num_workers,
    "start_text_token": start_text_token,
    "stop_text_token": stop_text_token,
    "max_text_len": max_text_len,
    "max_speech_len": max_speech_len,
    "prompt_duration": prompt_duration
}

# Create output folder structures if missing
os.makedirs('src', exist_ok=True)

# Write out parameters as an identical structured JSON backup file
with open('src/config.json', 'w') as json_file:
    json.dump(config_data, json_file, indent=4)

clear_output()
print("="*50)
print("\u2705 CONFIGURATION WRITTEN TO src/config.json")
print("="*50)
print(json.dumps(config_data, indent=2))


In [ ]:
# STEP 2: Clone Repository & Sync Configuration

import os

# Clone the repository
repo_url = "https://github.com/amazeble/chatterbox-finetuning.git"
print(f"Cloning repository from {repo_url}...")
!git clone {repo_url}
print("Repository cloned successfully.")

# Change directory into the cloned repo
os.chdir('chatterbox-finetuning')
print(f"Changed working directory to: {os.getcwd()}")

# Sync configuration to src/config.py
# Note: We exclude source_path and dest_path as they are Colab-specific
print("Syncing configuration to src/config.py...")

config_content = '''from dataclasses import dataclass, field
from typing import List, Literal, Optional
import os
import glob


def should_run_preprocessing(config) -> bool:
    """
    Determine if preprocessing should run based on config.preprocess setting.

    - If preprocess is True: always run
    - If preprocess is False: skip
    - If preprocess is "auto": check if preprocessed_dir exists and has same number of .pt files as .wav files

    Returns True if preprocessing should run, False otherwise.
    """
    if config.preprocess is True:
        return True
    elif config.preprocess is False:
        return False
    elif config.preprocess == "auto":
        # Check if preprocessed_dir exists
        if not os.path.exists(config.preprocessed_dir):
            return True

        # Count .wav files in wav_dir
        wav_files = glob.glob(os.path.join(config.wav_dir, "*.wav"))
        wav_count = len(wav_files)

        # Count .pt files in preprocessed_dir
        pt_files = glob.glob(os.path.join(config.preprocessed_dir, "*.pt"))
        pt_count = len(pt_files)

        # If counts don't match, rerun preprocessing
        if wav_count != pt_count:
            return True

        return False

    # Default to running preprocessing
    return True


@dataclass
class TrainConfig:
    # --- Paths ---
    # Directory where setup.py downloaded the files
    model_dir: str = "''' + model_dir + '''"

    # Path to your metadata CSV (Format: ID|RawText|NormText)
    csv_path: str = "''' + csv_path + '''"

    # Directory containing WAV files
    wav_dir: str = "''' + wav_dir + '''"

    preprocessed_dir = "''' + preprocessed_dir + '''"

    # Output directory for the finetuned model
    output_dir: str = "''' + output_dir + '''"

    is_inference = ''' + str(is_inference).lower() + '''
    inference_prompt_path: str = "''' + inference_prompt_path + '''"
    inference_test_text: str = "''' + inference_test_text + '''"


    ljspeech = ''' + str(ljspeech).lower() + ''' # Set True if the dataset format is ljspeech, and False if it's file-based.
    json_format = ''' + str(json_format).lower() + ''' # Set True if the dataset format is json, and False if it's file-based or ljspeech.
    # Preprocessing mode: True (always run), False (skip), "auto" (smart detection)
    preprocess: Optional[Literal[True, False, "auto"]] = ''' + (f'\"{preprocess}\"' if isinstance(preprocess, str) else str(preprocess).lower()) + '''

    is_turbo: bool = ''' + str(is_turbo).lower() + '''  # Set True if you're training Turbo, False if you're training Normal.
    is_lora: bool = ''' + str(is_lora).lower() + '''   # True: Efficient LoRA training (Recommended for < 10h data)
                           # False: Full Fine-Tune (High VRAM, for massive datasets)
    is_merge_lora: bool = ''' + str(is_merge_lora).lower() + '''  # If True and is_lora is True, automatically run merge_lora.py after training

    lora_r: int = ''' + str(lora_r) + '''
    lora_alpha: int = ''' + str(lora_alpha) + '''
    turbo_lora_target_modules: List[str] = field(default_factory=lambda: ''' + str(turbo_lora_target_modules) + ''')
    lora_target_modules: List[str] = field(default_factory=lambda: ''' + str(lora_target_modules) + ''')
    lora_modules_to_save: List[str] = field(default_factory=lambda: ''' + str(lora_modules_to_save) + ''')


    # --- Vocabulary ---
    # The size of the NEW vocabulary (from tokenizer.json)
    # Ensure this matches the JSON file generated by your tokenizer script.
    # For Turbo mode: Use the exact number provided by setup.py (e.g., 52260)
    new_vocab_size: int = ''' + str(new_vocab_size) + ''' if is_turbo else 2454

    # --- Hyperparameters ---
    batch_size: int = ''' + str(batch_size) + '''      # Adjust based on VRAM (2, 4, 8)
    grad_accum: int = ''' + str(grad_accum) + '''       # Effective Batch Size = Batch * Accum
    learning_rate: float = ''' + str(learning_rate) + ''' if is_lora else 1e-5  # T3 is sensitive, keep low
    num_epochs: int = ''' + str(num_epochs) + ''' if is_lora else 30

    save_steps: int = ''' + str(save_steps) + '''
    save_total_limit: int = ''' + str(save_total_limit) + '''
    dataloader_num_workers: int = ''' + str(dataloader_num_workers) + '''

    # --- Constraints ---
    start_text_token = 255
    stop_text_token = 0
    max_text_len: int = ''' + str(max_text_len) + '''
    max_speech_len: int = ''' + str(max_speech_len) + '''   # Truncates very long audio
    prompt_duration: float = ''' + str(prompt_duration) + ''' # Duration for the reference prompt (seconds)
'''

# Write the updated config file
with open('src/config.py', 'w') as f:
    f.write(config_content)

print("Configuration synced successfully to src/config.py")
print("\nUpdated configuration values:")
print(f"  model_dir: {model_dir}")
print(f"  csv_path: {csv_path}")
print(f"  wav_dir: {wav_dir}")
print(f"  preprocessed_dir: {preprocessed_dir}")
print(f"  output_dir: {output_dir}")
print(f"  is_turbo: {is_turbo}")
print(f"  is_lora: {is_lora}")
print(f"  is_merge_lora: {is_merge_lora}")
print(f"  lora_r: {lora_r}")
print(f"  lora_alpha: {lora_alpha}")
print(f"  batch_size: {batch_size}")
print(f"  learning_rate: {learning_rate}")
print(f"  num_epochs: {num_epochs}")



```
# This is formatted as code
```

## STEP 3: Google Drive Connection & Symlink Dataset

Mount Google Drive and create a symlink to sync the dataset from source_path to the Colab working directory.

In [ ]:
# STEP 3: Google Drive Connection & Symlink Dataset

from google.colab import drive
import os

# Mount Google Drive
print("Mounting Google Drive...")
drive.mount('/content/drive')

# Create destination directory if it doesn't exist
os.makedirs(dest_path, exist_ok=True)

# Change to destination directory
os.chdir(dest_path)
print(f"Changed working directory to: {dest_path}")

# Create symlink to dataset in Google Drive
print(f"Creating symlink from {source_path} to {dest_path}MyTTSDataset...")

# Check if source exists
if os.path.exists(source_path):
    # Create symlink path
    dest_dataset_path = os.path.join(dest_path, 'MyTTSDataset')
    
    # Remove existing symlink or directory if it exists
    if os.path.exists(dest_dataset_path) or os.path.islink(dest_dataset_path):
        if os.path.islink(dest_dataset_path):
            os.unlink(dest_dataset_path)
            print(f"Removed existing symlink at {dest_dataset_path}")
        elif os.path.isdir(dest_dataset_path):
            import shutil
            shutil.rmtree(dest_dataset_path)
            print(f"Removed existing directory at {dest_dataset_path}")
    
    # Create symbolic link
    os.symlink(source_path, dest_dataset_path)
    print(f"Symlink created successfully: {dest_dataset_path} -> {source_path}")
else:
    print(f"Warning: Source path {source_path} does not exist. Please check your Google Drive.")

# Verify the symlink
print("\nVerifying symlink:")
if os.path.islink(dest_dataset_path):
    print(f"  Symlink target: {os.readlink(dest_dataset_path)}")
    if os.path.exists(dest_dataset_path):
        for item in os.listdir(dest_dataset_path):
            item_path = os.path.join(dest_dataset_path, item)
            if os.path.isdir(item_path):
                count = len(os.listdir(item_path))
                print(f"  {item}/ ({count} items)")
            else:
                print(f"  {item}")
        print("\n✅ Symlink is valid and accessible!")
    else:
        print("❌ Warning: Symlink target is not accessible!")
elif os.path.exists(dest_dataset_path):
    print(f"  {dest_dataset_path} exists but is not a symlink")
else:
    print(f"  {dest_dataset_path} does not exist")
# Create one-way symlink from chatterbox_output to Google Drive
output_dir_local = "./chatterbox_output"
output_dir_gdrive = "/content/drive/MyDrive/chatterbox_output"

print(f"\nCreating one-way symlink from {output_dir_local} to {output_dir_gdrive}...")

# Create Google Drive output directory if it doesn't exist
os.makedirs(output_dir_gdrive, exist_ok=True)

# Remove existing symlink or directory if it exists
if os.path.exists(output_dir_local) or os.path.islink(output_dir_local):
    if os.path.islink(output_dir_local):
        os.unlink(output_dir_local)
        print(f"Removed existing symlink at {output_dir_local}")
    elif os.path.isdir(output_dir_local):
        import shutil
        shutil.rmtree(output_dir_local)
        print(f"Removed existing directory at {output_dir_local}")

# Create symbolic link pointing to Google Drive
os.symlink(output_dir_gdrive, output_dir_local)
print(f"Symlink created successfully: {output_dir_local} -> {output_dir_gdrive}")

# Verify the output symlink
print("\nVerifying output symlink:")
if os.path.islink(output_dir_local):
    print(f"  Symlink target: {os.readlink(output_dir_local)}")
    if os.path.exists(output_dir_local):
        items = os.listdir(output_dir_local)
        if items:
            print(f"  Contents ({len(items)} items):")
            for item in items[:5]:  # Show first 5 items
                print(f"    {item}")
            if len(items) > 5:
                print(f"    ... and {len(items) - 5} more")
        else:
            print("  (empty directory)")
        print("\n✅ Output symlink is valid and accessible!")
    else:
        print("❌ Warning: Output symlink target is not accessible!")
else:
    print(f"  {output_dir_local} does not exist or is not a symlink")


## STEP 4: Install Dependencies

Install system dependencies (ffmpeg) and Python requirements.

In [ ]:
# STEP 4: Install Dependencies

import os

# Update apt and install ffmpeg
print("Installing system dependencies (ffmpeg)...")
!sudo apt update
!sudo apt install -y ffmpeg

# Install Python requirements
print("\nInstalling Python requirements...")
!pip install peft==0.17.1 torch==2.6.0 torchaudio==2.6.0 torchvision==0.21.0 chatterbox-tts==0.1.2 silero-vad==6.2.0 librosa==0.11.0 soundfile==0.13.1 num2words ffmpeg-python tqdm pandas safetensors tensorboard omegaconf hf_transfer pyloudnorm gdown --index-url https://download.pytorch.org/whl/cu124

print("\nDependencies installed successfully.")

## STEP 5: Run Setup

Run the setup script to prepare pretrained models and other necessary components.

In [ ]:
# STEP 5: Run Setup

import os

print("Running setup.py...")
print(f"Working directory: {os.getcwd()}")

# Run setup.py
!python setup.py

print("\nSetup completed successfully.")

## STEP 6: Training & LoRA Merge

Run the training script. If `is_merge_lora` is True and `is_lora` is True, automatically merge LoRA weights after training.

In [ ]:
# STEP 6: Training & LoRA Merge

import os

print("Starting training...")
print(f"Working directory: {os.getcwd()}")
print(f"is_lora: {is_lora}")
print(f"is_merge_lora: {is_merge_lora}")

# Run training
!python train.py

# Check if we should merge LoRA
if is_lora and is_merge_lora:
    print("\n" + "="*50)
    print("LoRA training completed. Merging LoRA weights...")
    print("="*50 + "\n")
    !python merge_lora.py
    print("\nLoRA weights merged successfully.")
else:
    print("\nSkipping LoRA merge (is_lora=False or is_merge_lora=False).")

print("\n" + "="*50)
print("Training pipeline completed!")
print("="*50)

## Post-Training: Copy Results to Google Drive

Optional: Copy the trained model output back to Google Drive for safekeeping.

In [ ]:
# Optional: Copy trained model back to Google Drive

import shutil
import os

# Define output directory and Google Drive destination
output_dir_local = "./chatterbox_output"
output_dir_gdrive = "/content/drive/MyDrive/chatterbox_output"

if os.path.exists(output_dir_local):
    print(f"Copying trained model from {output_dir_local} to {output_dir_gdrive}...")

    # Remove existing directory in Google Drive if it exists
    if os.path.exists(output_dir_gdrive):
        shutil.rmtree(output_dir_gdrive)

    # Copy the output directory
    shutil.copytree(output_dir_local, output_dir_gdrive)
    print(f"Trained model copied successfully to {output_dir_gdrive}")
else:
    print(f"Output directory {output_dir_local} does not exist. Skipping copy.")